# Hello World with TorchTPU

This notebook demonstrates a simple text generation demo using `torch_tpu`. It loads a GPT-2 model and generates text on the selected device (TPU, GPU, or CPU).

In [ ]:
import time
import torch
from torch_tpu import api
import transformers

# Register TPU backend to PyTorch (requires TPU runtime)
api.tpu_device()

## Configure your device

To run on TPU instead of GPU, just swap your device to "tpu". No further changes are required.

In [ ]:
device = torch.accelerator.current_accelerator()
print(f"Current accelerator is: {torch.accelerator.current_accelerator()}")

## Initialize Model

In [ ]:
MODEL_PATH = "Qwen/Qwen3-0.6B"

# Load model and tokenizer
print(f"Downloading weights for {MODEL_PATH}. This may take a few minutes...")
start_load = time.time()

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH)
model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_PATH)

print(f"Finished {MODEL_PATH} in {time.time() - start_load:.4f}s")

## Move Model to Device

In [ ]:
print(f"Moving model to {device}...")
start_move = time.time()

model.to(device)

print(f"Moved in {time.time() - start_move:.4f}s")

## Generate Tokens

In [ ]:
INPUT_TEXT = "Hello, I am a"

# Run inference
print("Generating text...")
start_gen = time.time()

with torch.no_grad():
  inputs = tokenizer(INPUT_TEXT, return_tensors="pt").to(device)
  outputs = model.generate(
      **inputs,
      max_length=30,
      pad_token_id=tokenizer.pad_token_id,
  )

print(f"Generation took {time.time() - start_gen:.4f}s")

## Decode Tokens

In [ ]:
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Input: '{INPUT_TEXT}'")
print(f"Output: '{result}'")